# 16장 웹부록: 빛 곡선에서 외계행성 찾기

「영화관으로 간 AI 리터러시」 16장 **우주를 읽는 AI**의 코랩 실습입니다.

- 별의 밝기를 잰 데이터(빛 곡선)에 잡음을 섞어 직접 만듭니다.
- 잡음 속에 숨은 행성의 '홈'을 **접어서 보기**로 찾습니다.
- 가능한 주기를 모두 시험해 가장 또렷한 주기를 고릅니다.
- '행성'과 '가짜' 빛 곡선으로 간단한 **분류 모델**을 학습시킵니다.
- 덤으로 태양계 행성 데이터로 **케플러 제3법칙**의 직선을 그립니다.

코드를 몰라도 괜찮습니다. 위에서부터 셀을 하나씩 실행(▶)하세요. 설치나 인터넷 내려받기는 필요 없습니다.

In [ ]:
import os, numpy as np
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
# 그래프의 한글이 깨지지 않도록 나눔고딕을 씁니다 (코랩에서는 처음 한 번 설치, 10초쯤)
if os.path.exists('/content') and not os.path.exists('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
f = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(f):
    fm.fontManager.addfont(f); matplotlib.rc('font', family='NanumGothic')
matplotlib.rcParams['axes.unicode_minus'] = False
rng = np.random.default_rng(16)   # 16장이라서 16
plt.rcParams['figure.figsize'] = (10, 3.5)

## 1. 가짜 별빛 데이터 만들기

90일 동안 30분마다 별의 밝기를 쟀다고 가정합니다. 평소 밝기를 1.0으로 둡니다.

- 행성은 **10.3일**마다 별 앞을 지나고, 지날 때 빛이 **0.5%** 줄어듭니다.
- 망원경의 잡음은 0.2% 정도입니다. 홈보다 잡음이 더 커 보일 만큼 큽니다.

아래 숫자를 바꿔 가며 다시 실행해 보세요.

In [ ]:
PERIOD = 10.3      # 공전 주기(일)
DEPTH = 0.005      # 빛이 줄어드는 비율(0.5%)
DURATION = 0.3     # 별 앞을 지나는 시간(일)
NOISE = 0.002      # 잡음 크기

t = np.arange(0, 90, 0.5/24)                  # 30분 간격, 90일
phase0 = 2.7                                  # 첫 통과 시각
in_transit = ((t - phase0) % PERIOD) < DURATION
flux = 1.0 - DEPTH * in_transit + rng.normal(0, NOISE, t.size)

plt.plot(t, flux, '.', ms=2)
plt.xlabel('시간(일)'); plt.ylabel('밝기'); plt.title('원본 빛 곡선: 홈이 보이나요?')
plt.show()
print('측정값 개수:', t.size)

## 2. 접어서 보기

주기를 안다고 가정하고, 빛 곡선을 주기 길이로 잘라 겹칩니다. 진짜 홈은 매번 같은 자리에 있어 겹칠수록 또렷해지고, 잡음은 평평해집니다. 스마트폰 야간 촬영이 사진 여러 장을 겹치는 것과 같은 원리입니다.

In [ ]:
def fold(t, flux, period, nbins=100):
    ph = (t % period) / period
    bins = np.linspace(0, 1, nbins + 1)
    idx = np.digitize(ph, bins) - 1
    mean = np.array([flux[idx == k].mean() if (idx == k).any() else np.nan for k in range(nbins)])
    mean = np.where(np.isnan(mean), np.nanmedian(mean), mean)   # 빈 칸은 가운데값으로
    return ph, mean, (bins[:-1] + bins[1:]) / 2

ph, mean, centers = fold(t, flux, PERIOD)
plt.plot(ph, flux, '.', ms=2, alpha=0.3, label='모든 측정값')
plt.plot(centers, mean, 'r-', lw=2, label='구간 평균')
plt.xlabel('위상(한 바퀴 = 1)'); plt.ylabel('밝기'); plt.legend()
plt.title(f'{PERIOD}일로 접은 빛 곡선'); plt.show()

## 3. 주기를 모를 때: 모두 시험해 보기

1일부터 20일까지 0.01일 간격으로 주기를 바꿔 가며 접어 봅니다. 접었을 때 가장 깊은 홈이 생기는 주기가 정답 후보입니다. 계산이 2~3분쯤 걸릴 수 있습니다.

In [ ]:
periods = np.arange(1.0, 20.0, 0.01)
score = []
for P_ in periods:
    _, m, _ = fold(t, flux, P_, nbins=60)
    score.append(np.median(m) - m.min())   # 가장 깊은 홈의 깊이
score = np.array(score)
best = periods[score.argmax()]

plt.plot(periods, score)
plt.axvline(best, color='r', ls='--')
plt.xlabel('시험한 주기(일)'); plt.ylabel('홈의 깊이')
plt.title(f'가장 또렷한 주기: {best:.2f}일 (정답 {PERIOD}일)'); plt.show()

**생각해 보기**: 그래프에 정답의 절반, 두 배 자리에도 작은 봉우리가 있나요? 왜 그럴까요?

## 4. 홈의 깊이로 행성 크기 어림하기

빛이 줄어든 비율은 '행성 원판 넓이 ÷ 별 원판 넓이'입니다. 그래서 지름의 비율은 그 제곱근입니다. 태양 지름은 지구의 약 109배, 목성의 약 10배입니다.

In [ ]:
_, m, _ = fold(t, flux, best)
depth = np.median(m) - m.min()
ratio = np.sqrt(depth)
print(f'측정한 깊이: {depth*100:.2f}%')
print(f'행성 지름 / 별 지름 = {ratio:.3f}')
print(f'별이 태양만 하다면 행성 지름은 지구의 약 {ratio*109:.1f}배, 목성의 약 {ratio*10:.2f}배')

## 5. AI에게 '행성'과 '가짜'를 가르치기

실제 망원경 데이터에는 가짜 신호가 많습니다. 여기서는 세 종류를 만듭니다.

| 종류 | 모양 |
| --- | --- |
| 행성 | 바닥이 평평한 U자, 매번 깊이가 같음 |
| 쌍성 | V자 모양, 깊은 홈과 얕은 홈이 번갈아 나옴 |
| 잡음 | 뚜렷한 홈 없이 출렁임 |

주기의 두 배로 접은 빛 곡선(100칸)을 입력으로, 종류를 정답으로 주고 분류 모델을 학습시킵니다. 3장에서 배운 **지도학습**입니다.

In [ ]:
def make_example(kind):
    # 1번과 같은 방식으로 90일짜리 빛 곡선을 만들고, 주기의 2배로 접어 100칸으로 만듭니다
    P_ = rng.uniform(3, 15)
    dur = P_ * rng.uniform(0.02, 0.05)
    d = rng.uniform(0.002, 0.01)
    t0 = rng.uniform(0, P_)
    f = np.ones_like(t)
    if kind == 0:                                    # 행성: 매번 같은 깊이의 평평한 홈
        f -= d * (((t - t0) % P_) < dur)
    elif kind == 1:                                  # 쌍성: V자 홈, 한 번은 깊고 한 번은 얕음
        k = np.floor((t - t0) / P_) % 2
        x = ((t - t0) % P_) / dur
        v = np.clip(1 - np.abs(2 * x - 1), 0, None) * (x < 1)
        f -= np.where(k == 0, d, d * rng.uniform(0.2, 0.6)) * v
    else:                                            # 잡음: 별이 느리게 출렁임
        f += 0.003 * np.sin(2 * np.pi * t / rng.uniform(2, 20) + rng.uniform(0, 6))
    f += rng.normal(0, rng.uniform(0.001, 0.003), t.size)
    _, m, _ = fold(t - t0 + dur, f, 2 * P_, nbins=100)
    return (m - m.mean()) / (m.std() + 1e-9)

names = ['행성', '쌍성', '잡음']
X = np.array([make_example(k) for k in range(3) for _ in range(300)])   # 30초쯤 걸립니다
y = np.array([k for k in range(3) for _ in range(300)])

fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for k in range(3):
    ax[k].plot(X[y == k][0]); ax[k].set_title(names[k])
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = MLPClassifier(hidden_layer_sizes=(32,), max_iter=600, random_state=0)
model.fit(Xtr, ytr)
pred = model.predict(Xte)
print('시험 데이터 정확도:', round(accuracy_score(yte, pred) * 100, 1), '%')
print('혼동 행렬 (행 = 정답, 열 = 예측):')
print('        ' + '  '.join(names))
for i, row in enumerate(confusion_matrix(yte, pred)):
    print(names[i], row)

**생각해 보기**

- AI가 가장 많이 헷갈린 것은 무엇과 무엇인가요? 6장의 혼동 행렬을 떠올려 보세요.
- 1번에서 만든 진짜 빛 곡선을 접어서 이 모델에 넣으면 무엇이라고 할까요? 아래 셀로 확인해 보세요.
- 실제 연구에서는 AI가 '행성'이라고 해도 바로 발견으로 치지 않습니다. 어떤 확인이 더 필요할까요?

In [ ]:
_, mm, cc = fold(t, flux, best)
t0 = cc[np.argmin(mm)] * best - DURATION / 2      # 접은 곡선에서 홈이 시작하는 시각 어림
_, m2, _ = fold(t - t0 + DURATION, flux, 2 * best, nbins=100)
mine = (m2 - m2.mean()) / m2.std()
p = model.predict_proba([mine])[0]
for n, v in zip(names, p):
    print(f'{n}: {v*100:.1f}%')

## 6. 덤: 케플러 제3법칙을 데이터로 확인하기

태양까지의 평균 거리(지구 = 1)와 공전 주기(년)를 로그 눈금으로 그리면 한 직선 위에 섭니다. 기울기를 계산해 보세요. 케플러가 1618년에 찾은 값은 1.5입니다.

In [ ]:
planets = {'수성': (0.387, 0.241), '금성': (0.723, 0.615), '지구': (1.0, 1.0), '화성': (1.524, 1.881),
           '목성': (5.203, 11.86), '토성': (9.537, 29.46), '천왕성': (19.19, 84.01), '해왕성': (30.07, 164.8)}
a = np.array([v[0] for v in planets.values()]); T = np.array([v[1] for v in planets.values()])
slope, inter = np.polyfit(np.log10(a), np.log10(T), 1)
plt.figure(figsize=(6, 4.5))
plt.loglog(a, T, 'o')
for n, (ai, ti) in planets.items(): plt.annotate(n, (ai, ti), textcoords='offset points', xytext=(5, -10))
xs = np.logspace(-0.5, 1.6, 50); plt.loglog(xs, 10**inter * xs**slope, '--')
plt.xlabel('평균 거리(지구=1)'); plt.ylabel('공전 주기(년)')
plt.title(f'기울기 = {slope:.3f}'); plt.show()

**한 걸음 더**: 3장에서 배운 학습처럼, 컴퓨터가 데이터에서 '기울기 1.5'라는 규칙을 찾아낸 셈입니다. 하지만 '왜 1.5인가'를 설명한 것은 뉴턴의 만유인력입니다. 패턴 찾기와 이유 찾기의 차이를 생각해 보세요.